In [ ]:
from sagemaker.workflow.function_step import step
from sagemaker.workflow.pipeline import Pipeline
import sagemaker
from sagemaker.workflow.parameters import ParameterInteger
from sagemaker.workflow.execution_variables import ExecutionVariables
import utils

In [ ]:
role = sagemaker.get_execution_role()
sts = boto3.client("sts")
account_id = sts.get_caller_identity()["Account"]

region = boto3.Session().region_name
print(f"Region: {region}")
user = "grupo5"

default_bucket = "s3-mlflow-artifacts-mlflow-tracking-server-grupo5-01"
default_prefix = f"sagemaker/clients-attrition/{user}"

default_path = default_bucket + "/" + default_prefix
athena_database = "glue-catalog-database-utec-bank-01"

#Pipeline configuration
instance_type = "ml.m5.large"
pipeline_name = f"pipeline-inference-{user}"
model_name = f"model-attrition-{user}"
model_version = "latest"
# cod_month = ParameterInteger(name="PeriodoCarga")

#MLFlow configuration
tracking_server_arn = f"arn:aws:sagemaker:{region}:{account_id}:mlflow-tracking-server/{TRACKING_SERVER}"
experiment_name = f"pipeline-inference-exp-{user}"

In [ ]:
@step(
    name="DataPull",
    instance_type=instance_type,
    dependencies="./data_pull_requirements.txt"
)
def data_pull(experiment_name: str, run_name: str,
              cod_month: int) -> tuple[str, str, str]:
    import awswrangler as wr
    import mlflow

    mlflow.set_tracking_uri(tracking_server_arn)
    mlflow.set_experiment(experiment_name)
    query = """
       SELECT
        -- columnas de clientes
        c.id_correlativo AS id_cliente,
        c.codmes AS codmes_cliente,
        c.flg_bancarizado,
        c.rang_ingreso,
        c.flag_lima_provincia,
        c.edad,
        c.antiguedad,
        c.attrition,
        c.rang_sdo_pasivo_menos0,
        c.sdo_activo_menos0,
        c.sdo_activo_menos1,
        c.sdo_activo_menos2,
        c.sdo_activo_menos3,
        c.sdo_activo_menos4,
        c.sdo_activo_menos5,
        c.flg_seguro_menos0,
        c.flg_seguro_menos1,
        c.flg_seguro_menos2,
        c.flg_seguro_menos3,
        c.flg_seguro_menos4,
        c.flg_seguro_menos5,
        c.rang_nro_productos_menos0,
        c.flg_nomina,
        c.nro_acces_canal1_menos0,
        c.nro_acces_canal1_menos1,
        c.nro_acces_canal1_menos2,
        c.nro_acces_canal1_menos3,
        c.nro_acces_canal1_menos4,
        c.nro_acces_canal1_menos5,
        c.nro_acces_canal2_menos0,
        c.nro_acces_canal2_menos1,
        c.nro_acces_canal2_menos2,
        c.nro_acces_canal2_menos3,
        c.nro_acces_canal2_menos4,
        c.nro_acces_canal2_menos5,
        c.nro_acces_canal3_menos0,
        c.nro_acces_canal3_menos1,
        c.nro_acces_canal3_menos2,
        c.nro_acces_canal3_menos3,
        c.nro_acces_canal3_menos4,
        c.nro_acces_canal3_menos5,
        c.nro_entid_ssff_menos0,
        c.nro_entid_ssff_menos1,
        c.nro_entid_ssff_menos2,
        c.nro_entid_ssff_menos3,
        c.nro_entid_ssff_menos4,
        c.nro_entid_ssff_menos5,
        c.flg_sdo_otssff_menos0,
        c.flg_sdo_otssff_menos1,
        c.flg_sdo_otssff_menos2,
        c.flg_sdo_otssff_menos3,
        c.flg_sdo_otssff_menos4,
        c.flg_sdo_otssff_menos5,

        -- columnas de requerimientos (con alias si es necesario)
        r.codmes AS codmes_requerimiento,
        r.tipo_requerimiento2,
        r.dictamen,
        r.producto_servicio_2,
        r.submotivo_2

      FROM "glue-catalog-database-utec-bank-01"."clientes" AS c
      LEFT JOIN "glue-catalog-database-utec-bank-01"."requerimientos" AS r
        ON c.id_correlativo = r.id_correlativo
       WHERE   codmes = {}
    """.format(cod_month)

    inf_raw_s3_path = f"s3://{default_path}/inf-raw-data/{cod_month}.csv"
    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id
        with mlflow.start_run(run_name="DataPull", nested=True):
            df = wr.athena.read_sql_query(sql=query, database="risk_management")
            df.to_csv(inf_raw_s3_path, index=False)
            mlflow.log_input(
                mlflow.data.from_pandas(df, inf_raw_s3_path),
                context="DataPull"
            )
    return inf_raw_s3_path, experiment_name, run_id

In [ ]:
@step(
    name="ModelInference",
    instance_type=instance_type,
    dependencies="./model_training_requirements.txt"
)
def model_inference(inf_raw_s3_path: str, experiment_name: str,
                    run_id: str, cod_month: int) -> tuple[str, str, str]:
    import pandas as pd
    import mlflow
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import LabelEncoder

    mlflow.set_tracking_uri(tracking_server_arn)
    mlflow.set_experiment(experiment_name)
    
    # Same features used in training
    SELECTED_FEATURES = [
        'rang_sdo_pasivo_menos0', 'edad', 'antiguedad', 'rang_ingreso',
        'nro_acces_canal3_menos0', 'nro_acces_canal3_menos1', 'nro_acces_canal3_menos2',
        'nro_acces_canal3_menos3', 'rang_nro_productos_menos0', 'nro_entid_ssff_menos0',
        'nro_acces_canal3_menos4', 'nro_entid_ssff_menos5', 'sdo_activo_menos0',
        'nro_acces_canal3_menos5', 'nro_entid_ssff_menos4'
    ]
    
    model_uri = f"models:/{model_name}/{model_version}"
    df = pd.read_csv(inf_raw_s3_path)
    
    # Apply same preprocessing as training
    # Drop columns with many nulls
    df.drop(columns=[
        'tipo_requerimiento2', 'dictamen',
        'producto_servicio_2', 'submotivo_2'
    ], inplace=True, errors='ignore')
    
    # Imputation
    cat_imputer = SimpleImputer(strategy='most_frequent')
    df[['rang_ingreso', 'flag_lima_provincia']] = cat_imputer.fit_transform(
        df[['rang_ingreso', 'flag_lima_provincia']]
    )
    
    num_imputer = SimpleImputer(strategy='median')
    df[['edad', 'antiguedad']] = num_imputer.fit_transform(
        df[['edad', 'antiguedad']]
    )
    
    # Encoding
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    for col in cat_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
    
    # Select features for prediction
    X = df[SELECTED_FEATURES]
    
    # Load model and predict
    model = mlflow.pyfunc.load_model(model_uri)
    df["attrition_prob"] = model.predict_proba(X)[:, 1]
    df["attrition_prediction"] = model.predict(X)
    
    inf_proc_s3_path = f"s3://{default_path}/inf-proc-data/{cod_month}.csv"

    with mlflow.start_run(run_id=run_id):
        with mlflow.start_run(run_name="ModelInference", nested=True):
            df.to_csv(inf_proc_s3_path, index=False)
            mlflow.log_input(
                mlflow.data.from_pandas(df, inf_proc_s3_path),
                context="ModelInference"
            )
            mlflow.log_metric("inference_records", len(df))
            
    return inf_proc_s3_path, experiment_name, run_id

In [ ]:
@step(
    name="DataPush",
    instance_type=instance_type,
    dependencies="./data_pull_requirements.txt"
)
def data_push(inf_proc_s3_path: str, experiment_name: str, run_id: str, cod_month: int):
    
    import pandas as pd
    import mlflow
    import numpy as np
    from datetime import datetime
    import pytz
    import awswrangler as wr

    ID_COL = "id_cliente"
    TIME_COL = "codmes_cliente"
    PRED_COL = "attrition_prob"
    
    mlflow.set_tracking_uri(tracking_server_arn)
    mlflow.set_experiment(experiment_name)

    df = pd.read_csv(inf_proc_s3_path)
    
    # Create attrition risk profile based on probability
    df['attrition_profile'] = np.where(df[PRED_COL] >= 0.7, 'High risk',
                                      np.where(df[PRED_COL] >= 0.4, 'Medium risk',
                                      'Low risk'))

    df['model'] = model_name
    timezone = pytz.timezone("America/Lima")
    df['load_date'] = datetime.now(timezone).strftime("%Y%m%d")
    df['order'] = df[PRED_COL].rank(method='first', ascending=False).astype(int)

    inf_posproc_s3_path = f"s3://{default_path}/inf-posproc-data"
    inf_posproc_s3_path_partition = inf_posproc_s3_path + f'/{TIME_COL}={cod_month}/output.parquet'
    database = athena_database
    table_name = database + f'.attrition_predictions_{user.replace("-", "_")}'

    # Pushing data to S3 path
    df_output = df[[ID_COL, PRED_COL, 'attrition_prediction', 'model', 'attrition_profile', 'load_date', 'order', TIME_COL]] 
    df_output.to_parquet(inf_posproc_s3_path_partition, engine='pyarrow', compression='snappy')

    # Creating table
    ddl = f"""
    CREATE EXTERNAL TABLE IF NOT EXISTS {table_name} (
    {ID_COL} string,
    {PRED_COL} double,
    attrition_prediction int,
    model string,
    attrition_profile string,
    load_date string,
    order int
    )
    PARTITIONED BY ({TIME_COL} int)
    STORED AS parquet
    LOCATION '{inf_posproc_s3_path}'
    TBLPROPERTIES ('parquet.compression'='SNAPPY')
    """
    query_exec_id = wr.athena.start_query_execution(sql=ddl, database=database)
    wr.athena.wait_query(query_execution_id=query_exec_id)

    with mlflow.start_run(run_id=run_id):
        with mlflow.start_run(run_name="DataPush", nested=True):
            mlflow.log_input(
                mlflow.data.from_pandas(df_output, inf_posproc_s3_path_partition),
                context="DataPush"
            )
            mlflow.log_metric("records_pushed", len(df_output))
    
    # Refreshing partition
    dml = f"MSCK REPAIR TABLE {table_name}"
    query_exec_id = wr.athena.start_query_execution(sql=dml, database=database)
    wr.athena.wait_query(query_execution_id=query_exec_id)

In [ ]:
data_pull_step = data_pull(experiment_name=experiment_name,
                           run_name=ExecutionVariables.PIPELINE_EXECUTION_ID,
                           cod_month=cod_month)

model_inference_step = model_inference(inf_raw_s3_path=data_pull_step[0],
                                     experiment_name=data_pull_step[1],
                                     run_id=data_pull_step[2],
                                       cod_month=cod_month)

data_push_step = data_push(inf_proc_s3_path=model_inference_step[0],
                                     experiment_name=model_inference_step[1],
                                     run_id=model_inference_step[2],
                                      cod_month=cod_month)

In [ ]:
pipeline = Pipeline(name=pipeline_name,
                    steps=[data_pull_step, model_inference_step,data_push_step],
                    parameters=[cod_month])
pipeline.upsert(role_arn=role)

pipeline.start(parameters={"PeriodoCarga": 202501},
               execution_display_name="test-inference-full-1",
               execution_description="Testando inferece full 1")